# SULO Pizza Tutorial — Deployment and FAIRness

## Overview

An ontology that lives only on your laptop is of limited value to the scientific community. The **FAIR principles** — Findable, Accessible, Interoperable, Reusable — provide a framework for making digital resources (including ontologies) maximally useful:

| Principle | Applied to ontologies |
|---|---|
| **Findable** | The ontology has a stable, resolvable IRI; it is registered in a catalogue (e.g. BioPortal, OLS) |
| **Accessible** | The ontology IRI resolves to a downloadable file in a standard format |
| **Interoperable** | The ontology uses standard OWL constructs; it imports and aligns with upper-level ontologies |
| **Reusable** | The ontology carries a clear licence, is well-documented, and its terms have human-readable labels and definitions |

In this notebook we work through adding the metadata that makes the pizza ontology FAIR, publish it to a persistent IRI using the [OntoStart](https://github.com/micheldumontier/ontostart) CI/CD pipeline, and verify the result with the [FOOPS!](https://foops.linkeddata.es) API.

> **OntoStart** provides a ready-made GitHub repository structure — including GitHub Actions for validation, format conversion, documentation generation, and deployment — so you can bootstrap a FAIR ontology project in minutes.

## Learning objectives

By the end of this notebook you will be able to:

1. Add ontology-level metadata required by FAIR principles
2. Use `owl:versionIRI` and `owl:versionInfo` to version an ontology
3. Add the full set of metadata annotations used by the OntoStart pipeline (Dublin Core, VANN, PAV, DCAT, FOAF, MOD)
4. Audit and repair term-level `rdfs:label` and `rdfs:comment` annotations
5. Export an ontology to multiple standard serialisations (RDF/XML, Turtle)
6. Publish your ontology to a persistent, dereferenceable IRI using the OntoStart pipeline
7. Run an automated FOOPS! FAIRness assessment via the API and interpret the results

## Getting started

In [176]:
import sys, os
# Locate project root (the directory containing lib/) regardless of CWD
for _p in ['.', '..', '../..']:
    if os.path.isdir(os.path.join(_p, 'lib')):
        os.chdir(_p); sys.path.insert(0, os.getcwd()); break

from lib.helpers import *
import datetime
onto_path.append(".")

sulo  = get_ontology("dist/sulo.owl").load()
pro   = get_ontology("dist/pro.owl").load()
pizza = get_ontology("dist/pizza-06.owl").load()
pizza.imported_ontologies.append(sulo)
pizza.imported_ontologies.append(pro)
print("Pizza ontology IRI  :", pizza.base_iri)
print("Existing annotations:", pizza.metadata.comment)

Pizza ontology IRI  : https://w3id.org/ontostart/pizza/
Existing annotations: []


## Step 1 — Ontology IRI and Version IRI

Every FAIR ontology needs:
1. A **stable base IRI** — a permanent web address that identifies the ontology
2. A **version IRI** — an IRI that identifies this specific release

In OWL, the version IRI is declared with `owl:versionIRI`. The convention is:

```
Base IRI:    https://w3id.org/ontostart/pizza/
Version IRI: https://w3id.org/ontostart/pizza/releases/1.0.0/pizza.owl
```

The [W3ID](https://w3id.org) service provides permanent, redirectable IRIs — even if the server hosting the ontology changes, the W3ID IRI stays stable.

In [188]:
# owlready2 requires AnnotationProperties to be declared before they can be
# set on ontology metadata — even for OWL built-ins like owl:versionIRI.
# Declaring them in the owl namespace maps them to the correct IRIs in the saved file.

pizza_version = "1.0.0"
pizza_version_iri = f"https://w3id.org/ontostart/pizza/releases/{pizza_version}/pizza.owl"

with pizza:
    owl_ns = pizza.get_namespace("http://www.w3.org/2002/07/owl#")

    class versionIRI(AnnotationProperty):
        namespace = owl_ns

    class versionInfo(AnnotationProperty):
        namespace = owl_ns

    pizza.metadata.versionIRI = [pizza_version_iri]
    pizza.metadata.versionInfo = [pizza_version]

print("Version IRI :", pizza.metadata.versionIRI)
print("Version info:", pizza.metadata.versionInfo)

Version IRI : ['https://w3id.org/ontostart/pizza/releases/1.0.0/pizza.owl']
Version info: ['1.0.0']


## Step 2 — Ontology-Level Metadata

Ontology metadata is declared using standard vocabularies. The table below lists all fields supported by the [OntoStart](https://github.com/micheldumontier/ontostart) pipeline — filling these in ensures your ontology is correctly described when published.

### Core identification

| Property | Namespace | Purpose |
|---|---|---|
| `rdfs:label` | RDF Schema | Human-readable ontology name |
| `rdfs:comment` | RDF Schema | Short description of the ontology |

### Dublin Core

| Property | Namespace | Purpose |
|---|---|---|
| `dcterms:title` | DC Terms | Full title |
| `dcterms:description` | DC Terms | One-paragraph summary |
| `dcterms:alternative` | DC Terms | Short abbreviation (e.g. `"pizza"`) |
| `dc:creator` | DC Elements | Primary author — use an ORCID IRI |
| `dcterms:contributor` | DC Terms | Additional contributors |
| `dcterms:publisher` | DC Terms | Publishing organisation URL |
| `dcterms:license` | DC Terms | Licence URL (SPDX or Creative Commons) |
| `dcterms:created` | DC Terms | ISO 8601 creation date |
| `dcterms:issued` | DC Terms | ISO 8601 publication date |
| `dcterms:modified` | DC Terms | ISO 8601 last-modified date |
| `dcterms:language` | DC Terms | Language IRI (lexvo) |
| `dcterms:bibliographicCitation` | DC Terms | How to cite the ontology |

### Namespace, access, and provenance

| Property | Namespace | Purpose |
|---|---|---|
| `vann:preferredNamespacePrefix` | VANN | Suggested namespace prefix (e.g. `"pizza"`) |
| `vann:preferredNamespaceUri` | VANN | Namespace URI |
| `dcat:accessURL` | DCAT | URL where the ontology file can be downloaded |
| `foaf:homepage` | FOAF | Project or repository homepage |
| `pav:authoredBy` | PAV | Author IRI (complements `dc:creator`) |
| `schema:funding` | Schema.org | Funding acknowledgement text |

### Ontology registry metadata

| Property | Namespace | Purpose |
|---|---|---|
| `mod:status` | MOD | Lifecycle status: `"active"`, `"unstable"`, or `"deprecated"` |
| `mod:definitionProperty` | MOD | Which property carries term definitions (usually `rdfs:comment`) |
| `mod:prefLabelProperty` | MOD | Which property carries preferred labels (usually `rdfs:label`) |
| `mod:hasRepresentationLanguage` | MOD | Representation language (OWL) |
| `mod:hasSyntax` | MOD | Serialisation syntax |

In [189]:
import datetime

# ── Set your name (lowercase, no spaces) ──────────────────────────────────
YOUR_NAME = "smith"   # e.g. "jones", "dupont" — must match your ontostart branch
# ──────────────────────────────────────────────────────────────────────────

# ── Edit these values for your ontology ───────────────────────────────────
ONTO_ABBREV      = f"pizza-{YOUR_NAME}"
ONTO_IRI         = f"https://w3id.org/ontostart/{ONTO_ABBREV}/"
ONTO_TITLE       = "SULO Pizza Tutorial Ontology"
ONTO_DESCRIPTION = (
    "An OWL ontology for the pizza domain, built incrementally through the "
    "SULO Pizza Tutorial. Covers spatial composition, qualities, quantities, "
    "processes, information entities, time, and spatial containment."
)
ONTO_VERSION     = "1.0.0"
AUTHOR_ORCID     = "https://orcid.org/0000-0000-0000-0000"   # replace with your ORCID
PUBLISHER_URL    = "https://github.com/micheldumontier"
HOMEPAGE_URL     = "https://github.com/micheldumontier/ontostart"
CITATION         = "Unpublished"
FUNDING          = ""    # leave blank if not applicable
# ──────────────────────────────────────────────────────────────────────────

print(f"Ontology IRI : {ONTO_IRI}")
print(f"Abbrev       : {ONTO_ABBREV}")

now   = datetime.datetime.now(datetime.timezone.utc).isoformat()
today = datetime.date.today().isoformat()

with pizza:
    # ── Declare annotation properties by namespace ────────────────────────

    # DC Elements 1.1 (dc:)
    dc_ns = pizza.get_namespace("http://purl.org/dc/elements/1.1/")
    class dc_creator(AnnotationProperty):
        namespace = dc_ns
        python_name = "dc_creator"

    # DC Terms (dcterms:)
    dct_ns = pizza.get_namespace("http://purl.org/dc/terms/")
    for _name in ["title", "description", "alternative", "contributor",
                  "publisher", "license", "created", "issued", "modified",
                  "language", "bibliographicCitation"]:
        globals()[f"dct_{_name}"] = type(_name, (AnnotationProperty,), {"namespace": dct_ns})

    # PAV
    pav_ns = pizza.get_namespace("http://purl.org/pav/")
    class pav_authoredBy(AnnotationProperty):
        namespace = pav_ns
        python_name = "pav_authoredBy"

    # VANN
    vann_ns = pizza.get_namespace("http://purl.org/vocab/vann/")
    class vann_preferredNamespacePrefix(AnnotationProperty):
        namespace = vann_ns
        python_name = "vann_preferredNamespacePrefix"
    class vann_preferredNamespaceUri(AnnotationProperty):
        namespace = vann_ns
        python_name = "vann_preferredNamespaceUri"

    # DCAT
    dcat_ns = pizza.get_namespace("http://www.w3.org/ns/dcat#")
    class dcat_accessURL(AnnotationProperty):
        namespace = dcat_ns
        python_name = "dcat_accessURL"

    # FOAF
    foaf_ns = pizza.get_namespace("http://xmlns.com/foaf/0.1/")
    class foaf_homepage(AnnotationProperty):
        namespace = foaf_ns
        python_name = "foaf_homepage"

    # Schema.org
    schema_ns = pizza.get_namespace("https://schema.org/")
    class schema_funding(AnnotationProperty):
        namespace = schema_ns
        python_name = "schema_funding"

    # MOD
    mod_ns = pizza.get_namespace("https://w3id.org/mod#")
    for _name in ["status", "definitionProperty", "prefLabelProperty",
                  "hasRepresentationLanguage", "hasSyntax"]:
        globals()[f"mod_{_name}"] = type(_name, (AnnotationProperty,), {"namespace": mod_ns})

    # ── Set ontology-level annotations ───────────────────────────────────

    # Core identification
    pizza.metadata.label   = [ONTO_TITLE]
    pizza.metadata.comment = [ONTO_DESCRIPTION]

    # Dublin Core — essential
    pizza.metadata.title       = [ONTO_TITLE]
    pizza.metadata.description = [ONTO_DESCRIPTION]
    pizza.metadata.alternative = [ONTO_ABBREV]
    pizza.metadata.dc_creator  = [AUTHOR_ORCID]
    pizza.metadata.contributor = [AUTHOR_ORCID]
    pizza.metadata.publisher   = [PUBLISHER_URL]
    pizza.metadata.license     = ["https://creativecommons.org/licenses/by/4.0/"]
    pizza.metadata.created     = [now]
    pizza.metadata.issued      = [today]
    pizza.metadata.modified    = [now]
    pizza.metadata.language    = ["http://lexvo.org/id/iso639-1/en"]
    pizza.metadata.bibliographicCitation = [CITATION]

    # Namespace and access
    pizza.metadata.vann_preferredNamespacePrefix = [ONTO_ABBREV]
    pizza.metadata.vann_preferredNamespaceUri    = [ONTO_IRI]
    pizza.metadata.dcat_accessURL                = [ONTO_IRI]
    pizza.metadata.foaf_homepage                 = [HOMEPAGE_URL]
    pizza.metadata.pav_authoredBy                = [AUTHOR_ORCID]
    if FUNDING:
        pizza.metadata.schema_funding            = [FUNDING]

    # Ontology registry metadata — values are IRI strings, not Python objects
    pizza.metadata.status                    = ["active"]
    pizza.metadata.definitionProperty        = ["http://www.w3.org/2000/01/rdf-schema#comment"]
    pizza.metadata.prefLabelProperty         = ["http://www.w3.org/2000/01/rdf-schema#label"]
    pizza.metadata.hasRepresentationLanguage = ["http://omv.ontoware.org/2005/05/ontology#OWL"]
    pizza.metadata.hasSyntax                 = ["http://www.w3.org/ns/formats/Turtle"]

print("Metadata set. Key fields:")
print("  title  :", pizza.metadata.title)
print("  creator:", pizza.metadata.dc_creator)
print("  license:", pizza.metadata.license)
print("  version:", pizza.metadata.versionInfo)

Ontology IRI : https://w3id.org/ontostart/pizza-smith/
Abbrev       : pizza-smith
Metadata set. Key fields:
  title  : ['SULO Pizza Tutorial Ontology']
  creator: ['https://orcid.org/0000-0000-0000-0000']
  license: ['https://creativecommons.org/licenses/by/4.0/']
  version: ['1.0.0']


## Step 3 — Term-Level Annotations

FAIR also requires that **every term** in the ontology has at minimum:
- `rdfs:label` — a human-readable name, ideally with a language tag (`@en`)
- `rdfs:comment` — a textual definition

In owlready2, language-tagged labels are created with `locstr("text", "en")` and queried via `cls.label.en`. Let's audit how many pizza classes have English-tagged labels and definitions.

In [190]:
missing_label_en = []
missing_comment  = []

for cls in pizza.classes():
    if not cls.label.en:
        missing_label_en.append(cls.name)
    if not cls.comment:
        missing_comment.append(cls.name)

print(f"Classes missing rdfs:label[@en]: {len(missing_label_en)}")
print(f"Classes missing rdfs:comment   : {len(missing_comment)}")
if missing_label_en:
    print("  Missing @en labels:", missing_label_en[:10])
if missing_comment:
    print("  Missing comments  :", missing_comment[:10])

Classes missing rdfs:label[@en]: 1
Classes missing rdfs:comment   : 53
  Missing @en labels: ['OliveOil']
  Missing comments  : ['SpicySalamiPizza', 'SpicyPizza', 'OliveOil', 'SpicinessMeasurement', 'HotPepper', 'HotPepperPizza', 'Flour', 'Water', 'Sugar', 'BakersYeast']


In [191]:
import re

# Add missing English-tagged labels to classes that still lack one.
# Best practice: labels should be hand-curated; this auto-generates readable
# placeholders by splitting CamelCase names into words.
with pizza:
    for cls in pizza.classes():
        if not cls.label.en:
            readable = re.sub(r'(?<=[a-z])(?=[A-Z])', ' ', cls.name).lower()
            cls.label.append(locstr(readable, "en"))

n_labelled_en = sum(1 for c in pizza.classes() if c.label.en)
n_commented   = sum(1 for c in pizza.classes() if c.comment)
total = len(list(pizza.classes()))
print(f"Classes with rdfs:label[@en]: {n_labelled_en} / {total}")
print(f"Classes with rdfs:comment   : {n_commented} / {total} (definitions still mostly missing)")

Classes with rdfs:label[@en]: 71 / 71
Classes with rdfs:comment   : 18 / 71 (definitions still mostly missing)


## Step 4 — Exporting to Multiple Syntaxes

A FAIR ontology should be available in at least two standard serialisations so that different tools can consume it. The most common are:

- **RDF/XML** — the traditional OWL format; widest tool support (Protégé, ROBOT, OWLTools)
- **Turtle** — more human-readable; preferred for version-control diffs and linked-data workflows

owlready2 saves reliably in RDF/XML but its built-in Turtle serialiser produces an empty file. We use **rdflib** (already in `requirements.txt`) to load the saved RDF/XML and re-serialise it to Turtle.

In [192]:
import os
import rdflib
from rdflib.namespace import XSD, RDF, OWL

os.makedirs("dist", exist_ok=True)

# owlready2's built-in Turtle serialiser produces an empty file, so we:
#   1. save to RDF/XML (owlready2 native, reliable)
#   2. reload with rdflib and re-serialise to Turtle
pizza.save(file="dist/pizza.owl", format="rdfxml")

g = rdflib.Graph()
g.parse("dist/pizza.owl", format="xml")

# owlready2 auto-declares XSD datatypes as AnnotationProperties when date/time
# values are serialised. This violates OWL 2 DL ("reserved vocabulary").
# Strip any such declarations before saving.
for term in list(g.subjects(RDF.type, OWL.AnnotationProperty)):
    if str(term).startswith(str(XSD)):
        g.remove((term, RDF.type, OWL.AnnotationProperty))

g.serialize(destination="dist/pizza.ttl", format="turtle")

print("Exported:")
print("  dist/pizza.owl  (RDF/XML) —", os.path.getsize("dist/pizza.owl"), "bytes")
print("  dist/pizza.ttl  (Turtle)  —", os.path.getsize("dist/pizza.ttl"), "bytes")


Exported:
  dist/pizza.owl  (RDF/XML) — 76327 bytes
  dist/pizza.ttl  (Turtle)  — 43706 bytes


## Step 5 — Local FAIRness Pre-check

Before publishing, run a quick local self-assessment against the key FOOPS! indicators. This checks what we *can* verify locally (metadata completeness, label coverage, format exports). Indicators that require a live, dereferenceable URL (persistent URI resolution, content negotiation, HTML documentation) are verified in Step 7 using the real FOOPS! API.

In [193]:
# Self-assessment against FOOPS! indicators.
# Notes:
#   F1: checks that the IRI is an HTTP(S) URL — a local path would fail this.
#   A1: we exported both RDF/XML and Turtle above.
#   R1.4: checks for language-tagged @en labels specifically.
#   R1.4b: comment/definition check — still failing; exercise 1 addresses this.

n_classes = len(list(pizza.classes()))
checks = {
    "F1  — Ontology has an HTTP(S) IRI"                  : pizza.base_iri.startswith("http"),
    "F2  — Ontology has a version IRI"                   : bool(pizza.metadata.versionIRI),
    "A1  — Ontology serialised in a standard format"     : True,   # RDF/XML + Turtle saved above
    "I1  — Ontology uses OWL/RDF"                        : True,
    "I2  — Ontology re-uses terms from other ontologies" : any(pizza.imported_ontologies),
    "R1  — Ontology has a human-readable title"          : bool(pizza.metadata.title),
    "R1.1 — Ontology has a description"                  : bool(pizza.metadata.description),
    "R1.2 — Ontology has a licence"                      : bool(pizza.metadata.license),
    "R1.3 — Ontology has a creator"                      : bool(pizza.metadata.creator),
    "R1.4 — All classes have rdfs:label[@en]"            : all(c.label.en for c in pizza.classes()),
    "R1.4b — All classes have rdfs:comment"              : all(c.comment for c in pizza.classes()),
    "R1.5 — Ontology has a creation date"                : bool(pizza.metadata.created),
}

print("FOOPS! self-assessment")
print("-" * 55)
score = 0
for indicator, passed in checks.items():
    mark = "PASS" if passed else "FAIL"
    if passed: score += 1
    print(f"  [{mark}] {indicator}")

n_missing_en_label = sum(1 for c in pizza.classes() if not c.label.en)
n_missing_comment  = sum(1 for c in pizza.classes() if not c.comment)
print(f"\nScore: {score}/{len(checks)}")
if n_missing_en_label:
    print(f"  {n_missing_en_label}/{n_classes} classes still lack rdfs:label[@en]")
print(f"  {n_missing_comment}/{n_classes} classes still lack rdfs:comment — see Exercise 1")

FOOPS! self-assessment
-------------------------------------------------------
  [PASS] F1  — Ontology has an HTTP(S) IRI
  [PASS] F2  — Ontology has a version IRI
  [PASS] A1  — Ontology serialised in a standard format
  [PASS] I1  — Ontology uses OWL/RDF
  [PASS] I2  — Ontology re-uses terms from other ontologies
  [PASS] R1  — Ontology has a human-readable title
  [PASS] R1.1 — Ontology has a description
  [PASS] R1.2 — Ontology has a licence
  [FAIL] R1.3 — Ontology has a creator
  [PASS] R1.4 — All classes have rdfs:label[@en]
  [FAIL] R1.4b — All classes have rdfs:comment
  [PASS] R1.5 — Ontology has a creation date

Score: 10/12
  53/71 classes still lack rdfs:comment — see Exercise 1


In [194]:
pizza.save(file="dist/pizza-07.owl", format="rdfxml")
print("Ontology saved. Tutorial complete.")

Ontology saved. Tutorial complete.


## Step 6 — Publishing with OntoStart

The steps above produced a well-annotated, multi-format ontology file — but it still only lives on your laptop. To make it truly FAIR the file must be hosted at a stable, dereferenceable IRI.

[OntoStart](https://github.com/micheldumontier/ontostart) handles this automatically. When you push a branch to the OntoStart repository, a GitHub Actions pipeline:

1. **Validates** the ontology (OWL DL profile check, HermiT consistency)
2. **Converts** it to all standard serialisations (RDF/XML, Turtle, JSON-LD, N-Triples)
3. **Generates documentation** (Ontospy, PyLODE)
4. **Runs FOOPS!** and publishes a FAIRness badge
5. **Deploys to GitHub Pages**
6. **Serves the ontology** at `https://w3id.org/ontostart/{branch-name}/` with full content negotiation

The w3id.org redirect for `https://w3id.org/ontostart/` is already registered, so **no additional w3id.org pull request is needed** — your branch is published the moment the Actions run finishes.

### Your personal persistent IRI

Each tutorial attendee publishes their pizza ontology on a personal branch. The pattern is:

```
https://w3id.org/ontostart/pizza-{your-name}/
```

For example:
- `https://w3id.org/ontostart/pizza-smith/` (RDF/XML by default)
- `https://w3id.org/ontostart/pizza-smith/` with `Accept: text/turtle` header → Turtle
- `https://w3id.org/ontostart/pizza-smith/Pizza` → documentation page for the `Pizza` class

In [195]:
import shutil, subprocess

# YOUR_NAME and ONTO_ABBREV are set in the Step 2 cell above — run it first.
branch = ONTO_ABBREV   # e.g. "pizza-smith"
ttl_filename = f"{ONTO_ABBREV}.ttl"

print(f"Your branch name : {branch}")
print(f"Your ontology IRI: https://w3id.org/ontostart/{branch}/")
print()

# Copy the annotated ontology into the ontostart repo layout.
# Assumes the ontostart repo is cloned alongside this tutorial repo:
#   ../ontostart/
ontostart_dir = os.path.join("..", "ontostart")
if not os.path.isdir(ontostart_dir):
    print("ontostart repo not found at ../ontostart")
    print("Clone it first:  git clone https://github.com/micheldumontier/ontostart ../ontostart")
else:
    dest = os.path.join(ontostart_dir, ttl_filename)
    shutil.copy("dist/pizza.ttl", dest)
    print(f"Copied dist/pizza.ttl → {dest}")

    cmds = [
        ["git", "-C", ontostart_dir, "checkout", "-b", branch],
        ["git", "-C", ontostart_dir, "add", ttl_filename],
        ["git", "-C", ontostart_dir, "commit", "-m", f"add {branch} pizza ontology"],
        ["git", "-C", ontostart_dir, "push", "origin", branch],
    ]
    for cmd in cmds:
        result = subprocess.run(cmd, capture_output=True, text=True)
        label = " ".join(cmd[3:])
        if result.returncode == 0:
            print(f"  [OK] {label}")
        else:
            print(f"  [ERR] {label}")
            print(f"        {result.stderr.strip()}")

    print()
    print("GitHub Actions will now run. Check progress at:")
    print(f"  https://github.com/micheldumontier/ontostart/actions")
    print()
    print("Once the run completes, your ontology will be live at:")
    print(f"  https://w3id.org/ontostart/{branch}/")

Your branch name : pizza-smith
Your ontology IRI: https://w3id.org/ontostart/pizza-smith/

ontostart repo not found at ../ontostart
Clone it first:  git clone https://github.com/micheldumontier/ontostart ../ontostart


## Step 7 — FOOPS! Assessment

[FOOPS!](https://foops.linkeddata.es) evaluates 24 FAIRness indicators across all four FAIR dimensions.

### Option A — Upload your file (available now)

Use the [FOOPS! web validator](https://foops.linkeddata.es/FAIR_validator.html#) to upload `dist/pizza.ttl` directly. This works without deployment and gives immediate feedback on metadata completeness, label coverage, and vocabulary reuse.

### Option B — Assess via API (after deployment)

The FOOPS! REST API at `https://foops.linkeddata.es/assessOntology` requires a **publicly accessible URI** — it fetches the ontology from the web. Run the cell below against SULO first to see a well-scored result, then swap in your own pizza URI after Step 6 completes.

Some indicators can only pass once the ontology is live:
- **PURL1** — persistent URI scheme (w3id.org, purl, DOI…)
- **URI1 / VER2** — IRI resolves to a parseable RDF document
- **CN1** — server responds to RDF content negotiation
- **DOC1** — HTML documentation is served at the IRI
- **FIND3** — ontology is listed in BioPortal, LOV, or another catalogue

In [196]:
import urllib.request, urllib.parse, json

# ── Choose URI to assess ───────────────────────────────────────────────────
# Demo (before deployment): assess SULO to see a well-scored result
foops_uri = "https://w3id.org/sulo/sulo.owl"

# After Step 6 completes, replace with your own pizza ontology:
# foops_uri = f"https://w3id.org/ontostart/{ONTO_ABBREV}/"
# ──────────────────────────────────────────────────────────────────────────

print(f"Submitting to FOOPS!: {foops_uri}")
print("(this may take 30–60 seconds)")

payload = json.dumps({"ontologyUri": foops_uri}).encode()
req = urllib.request.Request(
    "https://foops.linkeddata.es/assessOntology",
    data=payload,
    headers={"Content-Type": "application/json", "Accept": "application/json"},
    method="POST",
)
with urllib.request.urlopen(req, timeout=120) as resp:
    result = json.loads(resp.read())

overall = result.get("overall_score", 0)
checks  = result.get("checks", [])

print(f"\nFOOPS! Assessment — {result.get('ontology_URI', foops_uri)}")
print(f"Overall score: {overall * 100:.1f}%  ({sum(1 for c in checks if c['status']=='ok')}/{len(checks)} checks passed)")
print(f"{'─' * 70}")

category_score = {}
for c in sorted(checks, key=lambda x: (x["category_id"], x["principle_id"])):
    cat  = c["category_id"]
    icon = "PASS" if c["status"] == "ok" else "FAIL"
    category_score.setdefault(cat, [0, 0])
    category_score[cat][1] += 1
    if c["status"] == "ok":
        category_score[cat][0] += 1
    print(f"  [{icon}] {c['abbreviation']:<12} {c['principle_id']:<6} {c['title'][:48]}")
    if c["status"] != "ok":
        print(f"         → {c['explanation'][:68]}")

print(f"{'─' * 70}")
print("\nBy FAIR category:")
for cat, (passed, total) in category_score.items():
    bar = "█" * passed + "░" * (total - passed)
    print(f"  {cat:<14} {bar}  {passed}/{total}")

print(f"\nFull browser report: https://foops.linkeddata.es/?ontURI={urllib.parse.quote(foops_uri)}")


Submitting to FOOPS!: https://w3id.org/sulo/sulo.owl
(this may take 30–60 seconds)

FOOPS! Assessment — https://w3id.org/sulo/
Overall score: 89.6%  (21/24 checks passed)
──────────────────────────────────────────────────────────────────────
  [PASS] CN1          A1     Ontology has content negotiation for RDF in RDF/
  [PASS] HTTP1        A1.1   Ontology uses an open protocol
  [PASS] FIND_3_BIS   A2     Ontology metadata are accessible, even when the 
  [PASS] PURL1        F1     Ontology has a persistent URL
  [PASS] URI1         F1     Ontology URI is resolvable
  [PASS] VER1         F1     A version IRI is declared in the ontology metada
  [PASS] VER2         F1     Ontology version IRI resolves
  [FAIL] URI2         F1     Consistent ontology IDs are employed
         → Ontology URI is different from ontology ID. Your ontologyURI (e.g., 
  [PASS] OM1          F2     Ontology minimum metadata is declared
  [PASS] FIND1        F3     Ontology prefix is declared
  [PASS] FIND2      

## Summary

In this notebook we made the pizza ontology FAIR-ready and published it:

| Step | What was added | FAIR dimension |
|---|---|---|
| 1 | `owl:versionIRI`, `owl:versionInfo` | Findable — F2 |
| 2 | Full metadata: DC, VANN, PAV, DCAT, FOAF, MOD | Reusable — R1 |
| 3 | `rdfs:label` for all classes; `rdfs:comment` audit | Reusable — R1.4 |
| 4 | Export to RDF/XML and Turtle | Accessible — A1 |
| 5 | Local FOOPS! pre-check | All |
| 6 | Published to `https://w3id.org/ontostart/pizza-{name}/` via OntoStart | All |
| 7 | Real FOOPS! API assessment on the live IRI | All |

Key takeaways:
- owlready2 requires annotation properties to be declared in the correct namespace before they can be set on ontology metadata — a library-level pattern, not an OWL requirement.
- The OntoStart pipeline reads `dc:`, `dcterms:`, `vann:`, `pav:`, `dcat:`, `foaf:`, `schema:`, and `mod:` metadata to generate documentation and FAIRness badges automatically.
- Auto-generated labels (CamelCase splitting) satisfy the label check but are not a substitute for hand-curated `rdfs:comment` definitions.
- OntoStart's branch-based publishing means a single w3id.org registration covers every branch — each attendee's personal ontology IRI costs only a `git push`.
- FOOPS! checks 24 indicators; those requiring a live, dereferenceable URL (content negotiation, HTML documentation, catalogue registration) can only be verified after deployment.

---

## Exercises

### Exercise 1 — Add term-level definitions

Choose five pizza classes that are still missing `rdfs:comment` definitions and add a meaningful one-sentence definition to each. Re-run the local FOOPS! pre-check (Step 5) and verify that the **R1.4b** indicator score improves.

In [ ]:
# Exercise 1 — your code here


### Exercise 2 — `owl:priorVersion`

OWL supports a `priorVersion` annotation to link a new release back to the previous one. Add a `priorVersion` annotation to the pizza ontology pointing to `https://w3id.org/ontostart/pizza/releases/0.9.0/pizza.owl`. How would this help users track the history of the ontology?

In [ ]:
# Exercise 2 — your code here


### Exercise 3 — Identify remaining FAIRness gaps

Our self-assessment shows a near-perfect local score, but real-world deployments expose additional gaps. For each item below, explain **(a)** what the gap is, **(b)** which FAIR principle it affects, and **(c)** how you would fix it:

1. Most classes still lack `rdfs:comment` definitions.
2. The version IRI `https://w3id.org/ontostart/pizza/releases/1.0.0/pizza.owl` is declared but not yet resolvable on the web.
3. There is no `owl:priorVersion` link to an earlier release.
4. The ontology has no `dcterms:contributor` or `dcterms:publisher` annotation.

*(Reflection — no code required.)*